# Agentic RAG: Router-Retriever System with PDF and Web Search Tools

**Course-End Project**

This notebook builds a two-agent system with **CrewAI**:

| Agent | Responsibility | Tools |
|---|---|---|
| **Router Agent** | Reads the user's question and decides the retrieval path: `PDF`, `WEB`, or `LLM` (direct answer, no retrieval) | None |
| **Retriever Agent** | Receives the router's decision, uses only the chosen tool, and writes a grounded answer with sources | `pdf_search` (CrewAI `PDFSearchTool`), `web_search` (`TavilySearchResults`) |

**Coordination:** a sequential CrewAI crew. The router task runs first; its structured decision is passed to the retriever task through `context=[router_task]`.

**Knowledge base (PDF path):** *Preliminary Report of the Independent International Scientific Panel on AI* (United Nations, July 2026) — an evidence-based assessment of AI capabilities, risks, impacts, governance and evidence gaps.

**Tracing:** every agent step and task result is written to `agent_trace_log.jsonl` and displayed at the end of the notebook.

> **Note on outputs:** the outputs shown are from the original run (October 2026). API keys have been removed; to re-run, create a `.env` file as described in the README.


## Step 1: Install required packages

In [1]:
%pip install -q crewai crewai-tools "crewai[azure-ai-inference]" litellm langchain langchain-openai langchain-community langchain-tavily langchain-classic tavily-python faiss-cpu pypdf tiktoken pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.


## Step 2: Import dependencies

In [2]:
import os
import json
import time
from datetime import datetime
from typing import Literal

import pandas as pd
from pydantic import BaseModel, Field

from crewai import Agent, Task, Crew, Process
from crewai.llm import LLM
from crewai.tools import BaseTool
from crewai.utilities.i18n import I18N_DEFAULT

# Neutralize the "force final answer" phrasing that triggers Azure OpenAI's prompt shield
# (same workaround as the course demo notebook).
I18N_DEFAULT._prompts["errors"]["force_final_answer"] = (
    "Please provide your best final answer based on the information retrieved so far."
)

## Step 3: Configuration and credentials

Credentials are read from a local `.env` file, which is never committed to Git. See `.env.example` for the variable names.

In [3]:
from dotenv import load_dotenv
load_dotenv()

AZURE_OPENAI_API_KEY = os.environ["AZURE_OPENAI_API_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_API_VERSION = os.environ["AZURE_OPENAI_API_VERSION"]
AZURE_OPENAI_DEPLOYMENT = os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"]
EMBEDDING_DEPLOYMENT = os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"]
# Note: in the original run this was first set to "text-embedding-3-small", which did not exist
# on the course gateway. Step 7b below records how the correct name was found and applied.

TAVILY_API_KEY = os.environ["TAVILY_API_KEY"]

# Several libraries read credentials from environment variables under different names
os.environ["OPENAI_API_VERSION"] = AZURE_OPENAI_API_VERSION
os.environ["AZURE_API_KEY"] = AZURE_OPENAI_API_KEY          # LiteLLM naming
os.environ["AZURE_API_BASE"] = AZURE_OPENAI_ENDPOINT
os.environ["AZURE_API_VERSION"] = AZURE_OPENAI_API_VERSION

# Locate the PDF anywhere under the project folder (same approach as the RAG project).
# Matches any PDF whose name starts with this stem, e.g. the "-compressed-pages-deleted" version.
PDF_NAME_STEM = "un_scientific_panel_ai_preliminary_report"
PDF_PATH = None
for root, dirs, files in os.walk("."):
    for f in files:
        if f.lower().endswith(".pdf") and f.startswith(PDF_NAME_STEM):
            PDF_PATH = os.path.join(root, f)
            break
    if PDF_PATH:
        break
print("PDF found:", PDF_PATH)
assert PDF_PATH, f"No PDF starting with '{PDF_NAME_STEM}' found - check the file name/location."
PDF_NAME = os.path.basename(PDF_PATH)

PDF found: ./un_scientific_panel_ai_preliminary_report_summary.pdf


### Diagnostic: find the embedding deployment name

The course gateway's embedding deployment name was not documented, and the first guess returned `404 DeploymentNotFound`. This cell tests candidate names. Several were tried during troubleshooting; the working one is kept here.

In [4]:
from langchain_openai import AzureOpenAIEmbeddings

candidates = ["text-embedding-ada-002"]
for name in candidates:
    try:
        AzureOpenAIEmbeddings(
            azure_endpoint=AZURE_OPENAI_ENDPOINT,
            api_key=AZURE_OPENAI_API_KEY,
            api_version=AZURE_OPENAI_API_VERSION,
            azure_deployment=name,
        ).embed_query("test")
        print(f"WORKS: {name}")
    except Exception as e:
        print(f"no: {name} -> {type(e).__name__}: {str(e)[:100]}")

WORKS: text-embedding-ada-002


## Step 4: Initialize the LLM (Azure GPT-5-mini via CrewAI/LiteLLM)

`max_completion_tokens` is raised to 4000. In the earlier demo the writer agent once returned an empty response at 1500; a possible cause is that a reasoning model can use part of the budget internally before producing visible output.

In [5]:
llm = LLM(
    model=f"azure/{AZURE_OPENAI_DEPLOYMENT}",
    api_key=AZURE_OPENAI_API_KEY,
    base_url=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
    is_litellm=True,
    temperature=0,
    max_completion_tokens=4000,
)
print("LLM ready:", AZURE_OPENAI_DEPLOYMENT)

LLM ready: gpt-5-mini


## Step 5: Interaction logger (reasoning traceability)

Two CrewAI hooks feed one log file:
- `step_callback` — fires on each agent step (thought, tool call, tool result, final answer)
- `task_callback` — fires when a task (router or retriever) completes

Each entry is appended to `agent_trace_log.jsonl` and kept in memory for display.

In [6]:
LOG_FILE = "agent_trace_log.jsonl"
TRACE = []                      # in-memory copy of all log entries
CURRENT_RUN = {"run_id": None, "question": None}

def _short(text, n=600):
    text = "" if text is None else str(text)
    return text if len(text) <= n else text[:n] + " ...[truncated]"

def log_event(event_type, agent=None, detail=None, extra=None):
    entry = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "run_id": CURRENT_RUN["run_id"],
        "question": CURRENT_RUN["question"],
        "event": event_type,
        "agent": agent,
        "detail": _short(detail),
    }
    if extra:
        entry.update(extra)
    TRACE.append(entry)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

def step_logger(step):
    """Called by CrewAI after every agent step. Step objects differ by type, so read fields defensively."""
    tool = getattr(step, "tool", None)
    if tool:
        log_event("tool_call",
                  detail=f"thought: {getattr(step, 'thought', '')}",
                  extra={"tool": tool,
                         "tool_input": _short(getattr(step, "tool_input", ""), 300),
                         "tool_output": _short(getattr(step, "result", ""), 400)})
    elif hasattr(step, "output"):
        log_event("agent_final_answer", detail=getattr(step, "output", ""))
    else:
        log_event("agent_step", detail=step)

def task_logger(task_output):
    """Called by CrewAI when each task completes."""
    log_event("task_completed",
              agent=getattr(task_output, "agent", None),
              detail=getattr(task_output, "raw", task_output))

# Start each notebook run with a clean log
open(LOG_FILE, "w").close()
print("Logging to", LOG_FILE)

Logging to agent_trace_log.jsonl


## Step 6: PDF search tool (CrewAI `PDFSearchTool`, with fallback)

**Primary:** `crewai_tools.PDFSearchTool` pointed at the UN report and configured for Azure embeddings, wrapped in a `pdf_search` tool so it can log which backend answered.

**Fallback:** CrewAI's documentation does not give a complete Azure embedding example for `PDFSearchTool`, and community reports show it failing with Azure. The course gateway also rate-limits embedding requests. If `PDFSearchTool` fails, the tool automatically switches to a FAISS vector store (batched embedding, saved to disk so it is built only once). The backend actually used is logged.

In the original run, `PDFSearchTool` failed with a rate-limit error (below), so all PDF answers came from the FAISS fallback.

In [7]:
from crewai_tools import PDFSearchTool

USE_CREWAI_PDFSEARCHTOOL = True     # set False to go straight to the FAISS fallback

pdf_tool = None
PDF_BACKEND = "faiss"

if USE_CREWAI_PDFSEARCHTOOL:
    try:
        azure_embed_config = {
            "embedding_model": {
                "provider": "azure",
                "config": {
                    "model": EMBEDDING_DEPLOYMENT,
                    "deployment_id": EMBEDDING_DEPLOYMENT,
                    "api_key": AZURE_OPENAI_API_KEY,
                    "api_base": AZURE_OPENAI_ENDPOINT,
                    "api_version": AZURE_OPENAI_API_VERSION,
                },
            }
        }
        print(f"Indexing with PDFSearchTool: {PDF_NAME} ...")
        pdf_tool = PDFSearchTool(pdf=PDF_PATH, config=azure_embed_config)
        PDF_BACKEND = "crewai_pdfsearchtool"
        print("PDFSearchTool ready.")
    except Exception as e:
        print(f"PDFSearchTool setup failed -> using FAISS fallback.\nReason: {type(e).__name__}: {e}")
        pdf_tool = None

print("PDF backend:", PDF_BACKEND)

Indexing with PDFSearchTool: un_scientific_panel_ai_preliminary_report_summary.pdf ...
PDFSearchTool setup failed -> using FAISS fallback.
Reason: RateLimitError: Error code: 429 - {'statusCode': 429, 'message': 'Token limit will exceed based on estimated request tokens. Try again in 60 seconds.'} in upsert.
PDF backend: faiss


In [8]:
# ---- FAISS fallback (reuses the working approach from the RAG project) ----
FAISS_DIR = "faiss_index_un_report"
_faiss_store = None

def get_faiss_store():
    """Build (once) or load the FAISS index of the UN report."""
    global _faiss_store
    if _faiss_store is not None:
        return _faiss_store

    from langchain_openai import AzureOpenAIEmbeddings
    from langchain_community.vectorstores import FAISS
    from langchain_community.document_loaders import PyPDFLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    embeddings = AzureOpenAIEmbeddings(
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        api_key=AZURE_OPENAI_API_KEY,
        api_version=AZURE_OPENAI_API_VERSION,
        azure_deployment=EMBEDDING_DEPLOYMENT,
    )

    if os.path.isdir(FAISS_DIR):
        _faiss_store = FAISS.load_local(FAISS_DIR, embeddings, allow_dangerous_deserialization=True)
        print(f"Loaded saved FAISS index ({_faiss_store.index.ntotal} vectors).")
        return _faiss_store

    documents = PyPDFLoader(PDF_PATH).load()
    chunks = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200).split_documents(documents)

    # Batched embedding to stay under the rate limit (same settings as the RAG project)
    batch_size, pause = 5, 15
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]
        if _faiss_store is None:
            _faiss_store = FAISS.from_documents(batch, embeddings)
        else:
            _faiss_store.add_documents(batch)
        print(f"Embedded {min(i + batch_size, len(chunks))}/{len(chunks)} chunks")
        time.sleep(pause)

    _faiss_store.save_local(FAISS_DIR)
    print(f"FAISS index saved to {FAISS_DIR}/")
    return _faiss_store

def faiss_search(query, k=4):
    store = get_faiss_store()
    results = store.similarity_search(query, k=k)
    out = []
    for d in results:
        src = os.path.basename(d.metadata.get("source", "unknown"))
        page = d.metadata.get("page", "?")
        page = page + 1 if isinstance(page, int) else page   # PyPDFLoader pages are 0-based
        out.append(f"[Source: {src}, page {page}]\n{d.page_content}")
    return "\n\n---\n\n".join(out)

In [9]:
class ReportsPDFSearchTool(BaseTool):
    name: str = "pdf_search"
    description: str = (
        "Search the static PDF: the UN Independent International Scientific Panel on AI "
        "Preliminary Report (July 2026), covering AI capabilities and adoption, risks, "
        "economic, security, environmental and human-rights impacts, governance, evidence "
        "gaps and the Panel's next steps. Input: a search query string. "
        "Returns relevant passages with document and page."
    )

    def _run(self, query: str) -> str:
        global PDF_BACKEND
        if PDF_BACKEND == "crewai_pdfsearchtool" and pdf_tool is not None:
            try:
                result = f"[Source: {PDF_NAME}]\n{pdf_tool.run(query=query)}"
                log_event("pdf_backend", detail=f"PDFSearchTool answered query: {query}",
                          extra={"backend": PDF_BACKEND})
                return result
            except Exception as e:
                log_event("pdf_backend", detail=f"PDFSearchTool query failed ({e}); switching to FAISS",
                          extra={"backend": "faiss"})
                PDF_BACKEND = "faiss"
        result = faiss_search(query)
        log_event("pdf_backend", detail=f"FAISS answered query: {query}", extra={"backend": "faiss"})
        return result

pdf_search_tool = ReportsPDFSearchTool()

## Step 7: Web search tool (`TavilySearchResults`)

`TavilySearchResults` is a LangChain tool, so it is wrapped in a CrewAI `BaseTool` (the same pattern as the course demo). If your installed LangChain version no longer provides it, the cell falls back to its replacement, `TavilySearch` from `langchain-tavily`, and says so.

In [10]:
try:
    from langchain_community.tools.tavily_search import TavilySearchResults
    tavily = TavilySearchResults(max_results=4)
    WEB_BACKEND = "TavilySearchResults"
except Exception as e:
    print(f"TavilySearchResults unavailable ({e}); using langchain_tavily.TavilySearch instead.")
    from langchain_tavily import TavilySearch
    tavily = TavilySearch(max_results=4)
    WEB_BACKEND = "TavilySearch"

class WebSearchTool(BaseTool):
    name: str = "web_search"
    description: str = (
        "Search the internet for current or recent information that is not in the static reports. "
        "Input: a search query string. Returns titles, URLs and content snippets."
    )

    def _run(self, query: str) -> str:
        raw = tavily.invoke({"query": query})
        # TavilySearchResults returns a list of dicts; TavilySearch returns a dict with "results"
        results = raw.get("results", []) if isinstance(raw, dict) else raw
        lines = []
        for r in results:
            if isinstance(r, dict):
                lines.append(f"[Source: {r.get('title', 'No title')} - {r.get('url', 'No URL')}]\n"
                             f"{r.get('content', '')}")
            else:
                lines.append(str(r))
        log_event("web_backend", detail=f"{WEB_BACKEND} query: {query}", extra={"backend": WEB_BACKEND})
        return "\n\n---\n\n".join(lines) if lines else "No web results found."

web_search_tool = WebSearchTool()
print("Web search backend:", WEB_BACKEND)

Web search backend: TavilySearchResults


/tmp/ipykernel_15466/2829625514.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults
/tmp/ipykernel_15466/2829625514.py:3: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily = TavilySearchResults(max_results=4)


### Step 7b: Troubleshooting record — apply the correct embedding name

In the original run, Step 3 still held the wrong embedding name (`text-embedding-3-small`, shown in the first output below), and the kernel kept that value in memory. These two cells confirmed the problem, set the working name (`text-embedding-ada-002`), cleared any cached index, and tested the PDF search on its own. With the name set correctly in `.env`, they simply re-confirm the setting.

In [11]:
print("Embedding name in use:", EMBEDDING_DEPLOYMENT)
print("PDF backend:", PDF_BACKEND)

Embedding name in use: text-embedding-3-small
PDF backend: faiss


In [13]:
# Apply the correct embedding name and clear any old PDF index from memory
EMBEDDING_DEPLOYMENT = os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"]   # text-embedding-ada-002 in the original run
_faiss_store = None
print("Embedding name in use:", EMBEDDING_DEPLOYMENT)

# Test the PDF search on its own
print(pdf_search_tool.run(query="evidence gaps identified by the Panel")[:1200])

Embedding name in use: text-embedding-ada-002
Embedded 5/9 chunks
Embedded 9/9 chunks
FAISS index saved to faiss_index_un_report/
[Source: un_scientific_panel_ai_preliminary_report_summary.pdf, page 4]
to shape, evaluate and deploy AI. This preliminary report is itself part of the Panel’s contribution, a shared evidence 
base for Member States navigating increasingly urgent decisions. As the Panel’s understanding deepens through 
continued engagement with Member States and the broader scientific community, so too will its analysis, expanding 
beyond the gaps identified to chart the trajectories, tensions and opportunities that will define the future of AI.

---

[Source: un_scientific_panel_ai_preliminary_report_summary.pdf, page 4]
requires dedicated investments and policies to incentivize equitable access and reward innovation, while preventing 
the exploitation of vulnerable populations, particularly children, and avoiding displacement of expertise, psychological 
dependency or cult

## Step 8: Smoke-test both tools before involving the agents

In [14]:
print("=== PDF tool ===")
print(pdf_search_tool.run(query="evidence gaps identified by the Panel")[:1200])
print("\n=== Web tool ===")
print(web_search_tool.run(query="EU AI Act enforcement news this week")[:1200])

=== PDF tool ===
[Source: un_scientific_panel_ai_preliminary_report_summary.pdf, page 4]
to shape, evaluate and deploy AI. This preliminary report is itself part of the Panel’s contribution, a shared evidence 
base for Member States navigating increasingly urgent decisions. As the Panel’s understanding deepens through 
continued engagement with Member States and the broader scientific community, so too will its analysis, expanding 
beyond the gaps identified to chart the trajectories, tensions and opportunities that will define the future of AI.

---

[Source: un_scientific_panel_ai_preliminary_report_summary.pdf, page 4]
requires dedicated investments and policies to incentivize equitable access and reward innovation, while preventing 
the exploitation of vulnerable populations, particularly children, and avoiding displacement of expertise, psychological 
dependency or cultural and linguistic erasure.
Policymakers seeking to shape this governance face an evidence dilemma: they need ev

## Step 9: Define the agents

- **Router Agent** has no tools. Its only job is classification, and its output is forced into a structured `RouteDecision` (Pydantic model), so the decision is machine-readable and easy to log.
- **Retriever Agent** has both tools but is instructed to use only the one the router chose.

In [15]:
class RouteDecision(BaseModel):
    route: Literal["PDF", "WEB", "LLM"] = Field(description="Chosen retrieval path")
    reason: str = Field(description="One-sentence justification for the choice")
    search_query: str = Field(description="Concise query to send to the chosen tool (empty for LLM)")

router_agent = Agent(
    role="Query Router",
    goal="Classify each user question and choose the single best source to answer it: PDF, WEB, or LLM.",
    backstory=(
        "You are a triage specialist for a research desk. The desk's static library holds one "
        "document: the Preliminary Report of the UN Independent International Scientific Panel "
        "on AI (July 2026). It covers AI capabilities and adoption, concentration of AI "
        "development, the AI divide, agentic AI, information integrity, human and children's "
        "rights, economic, security and environmental implications, governance, evidence gaps "
        "and the Panel's next steps.\n"
        "Routing rules:\n"
        "- PDF: the question asks what this report or the Panel says, finds or recommends, or "
        "concerns the topics above as assessed by the Panel.\n"
        "- WEB: the question needs current, recent or real-time information (news, events "
        "after July 2026, prices, named products or people), or is outside the report's scope.\n"
        "- LLM: a general-knowledge or definitional question that needs no retrieval.\n"
        "You never answer the question yourself; you only route it."
    ),
    llm=llm,
    allow_delegation=False,
    verbose=True,
)

retriever_agent = Agent(
    role="Retriever and Answer Writer",
    goal="Retrieve evidence using ONLY the tool chosen by the Router and write an accurate, source-grounded answer.",
    backstory=(
        "You are a careful research analyst. You follow the router's decision exactly: "
        "PDF -> use pdf_search; WEB -> use web_search; LLM -> answer from your own knowledge "
        "without tools. You only state what the retrieved evidence supports, cite the source "
        "document or URL for each key point, and say clearly when the evidence is insufficient."
    ),
    tools=[pdf_search_tool, web_search_tool],
    llm=llm,
    allow_delegation=False,
    max_iter=5,
    verbose=True,
)

## Step 10: Define the tasks and the crew (sequential coordination)

Input/output flow:
1. `{question}` → **Router task** → `RouteDecision` JSON
2. `RouteDecision` (via `context=[router_task]`) + `{question}` → **Retriever task** → final grounded answer

A fresh crew is built for each question so every run is independent and gets its own trace ID.

In [16]:
def build_crew():
    router_task = Task(
        description=(
            "User question: {question}\n\n"
            "Decide whether this question should be answered from the PDF reports (PDF), "
            "a web search (WEB), or directly by the language model without retrieval (LLM). "
            "Also write a concise search query for the chosen tool."
        ),
        expected_output="A RouteDecision with route (PDF, WEB or LLM), a one-sentence reason, and a search_query.",
        output_pydantic=RouteDecision,
        agent=router_agent,
    )

    retriever_task = Task(
        description=(
            "User question: {question}\n\n"
            "Read the Router's decision in your context and follow it strictly:\n"
            "- route PDF -> call pdf_search with the router's search_query (you may refine it once).\n"
            "- route WEB -> call web_search with the router's search_query (you may refine it once).\n"
            "- route LLM -> do NOT call any tool; answer from general knowledge.\n"
            "Do not use the other tool. Write the final answer in 150-250 words, grounded only in "
            "the retrieved evidence, and finish with a 'Sources:' list (document names with pages, or URLs). "
            "Begin the answer with a line 'Route used: <PDF|WEB|LLM>'."
        ),
        expected_output="'Route used: ...' line, a grounded answer of 150-250 words, then a 'Sources:' list.",
        context=[router_task],
        agent=retriever_agent,
    )

    crew = Crew(
        agents=[router_agent, retriever_agent],
        tasks=[router_task, retriever_task],
        process=Process.sequential,
        step_callback=step_logger,
        task_callback=task_logger,
        verbose=True,
    )
    return crew


RESULTS = []   # summary of each demo run

async def ask(question: str):
    """Run one question through Router -> Retriever and print a clean summary."""
    CURRENT_RUN["run_id"] = f"run_{len(RESULTS) + 1}"
    CURRENT_RUN["question"] = question
    log_event("question_received", detail=question)

    crew = build_crew()
    result = await crew.kickoff_async(inputs={"question": question})

    decision = result.tasks_output[0].pydantic
    route = decision.route if decision else "UNPARSED"
    log_event("route_decision", agent="Query Router",
              detail=decision.reason if decision else result.tasks_output[0].raw,
              extra={"route": route, "search_query": decision.search_query if decision else None})

    answer = result.raw
    log_event("final_answer", agent="Retriever and Answer Writer", detail=answer)
    RESULTS.append({"run_id": CURRENT_RUN["run_id"], "question": question,
                    "route": route, "reason": decision.reason if decision else "",
                    "answer": answer})

    print("\n" + "=" * 90)
    print(f"QUESTION : {question}")
    print(f"ROUTE    : {route}  ({decision.reason if decision else 'no structured decision'})")
    print("-" * 90)
    print(answer)
    print("=" * 90)
    return result

## Step 11: Demonstration

Three questions, one per route. Expected routing:

| # | Question | Expected route |
|---|---|---|
| 1 | Evidence gaps identified in the UN Panel's report | **PDF** |
| 2 | This week's EU AI regulation news | **WEB** |
| 3 | What is retrieval-augmented generation? | **LLM** |

### Demo 1 — PDF route

In [17]:
await ask("According to the UN Scientific Panel's preliminary report, what are the main evidence gaps where the Panel cannot yet draw confident conclusions?")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 00d3b7a2-41c4-4df4-8586-8db0e510b288                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: User question: According to the UN Scientific Panel's preliminary report, what are the main evidence     │
│  gaps where the Panel cannot yet draw confident conclusions?                                                    │
│                                                                                                                 │
│  Decide whether this question should be answered from the PDF reports (PDF), a web search (WEB), or directly    │
│  by the language model without retrieval (LLM). Also write a concise search query for the chosen tool.          │
│  ID: 8948112c-8937-4429-9951-ea0c97b579da                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│  Task: User question: According to the UN Scientific Panel's preliminary report, what are the main evidence     │
│  gaps where the Panel cannot yet draw confident conclusions?                                                    │
│                                                                                                                 │
│  Decide whether this question should be answered from the PDF reports (PDF), a web search (WEB), or directly    │
│  by the language model without retrieval (LLM). Also write a concise search query for the chosen tool.          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {"route":"PDF","reason":"The user asks specifically what the UN Scientific Panel's preliminary report states   │
│  about evidence gaps, which is directly answered by the Panel’s PDF report.","search_query":"UN Independent     │
│  International Scientific Panel on AI Preliminary Report July 2026 evidence gaps where Panel cannot draw        │
│  confident conclusions"}                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Retriever and Answer Writer                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Route used: PDF                                                                                                │
│                                                                                                                 │
│  According to the Panel’s preliminary report, the retrieved text supports these principal evidence gaps where   │
│  the Panel cannot yet offer confident, fully quantified conclusions:                                            │
│                                                                                                                 │
│  - Evidence lags the pace of AI development, creating an “evidence dilemma” for timely governance decisions     │
│  (un_scientific_panel_ai_preliminary_report_summary.pdf, p. 4).                                                 │
│  - Evaluation methods are underdeveloped and many governance instruments are fragmented and “rarely measure     │
│  real‑world effectiveness,” limiting assessment of policy impact                                                │
│  (un_scientific_panel_ai_preliminary_report_summary.pdf, p. 4).                                                 │
│  - Institutions needed for independent capability and risk assessment remain embryonic, constraining            │
│  authoritative evaluation (un_scientific_panel_ai_preliminary_report_summary.pdf, p. 4).                        │
│  - Capacity is unevenly distributed: most countries lack technical expertise to assess the most capable         │
│  “frontier” models; compute, evaluation expertise and relevant data are concentrated where AI is built          │
│  (un_scientific_panel_ai_preliminary_report_summary.pdf, p. 4).                                                 │
│  - Technical limitations persist (reliability, cross‑language/culture performance, physical interaction,        │
│  multi‑step project execution, factual accuracy), creating uncertainty about real‑world behaviour and impacts   │
│  (un_scientific_panel_ai_preliminary_report_summary.pdf, p. 2).                                                 │
│                                                                                                                 │
│  The retrieved excerpts do not present a formal, enumerated list stating exactly which hypotheses the Panel     │
│  “cannot yet draw confident conclusions” about; the evidence is therefore insufficient to produce such a        │
│  verbatim list beyond the broad gaps above (un_scientific_panel_ai_preliminary_report_summary.pdf, p. 4).       │
│                                                                                                                 │
│  Sources:                                                                                                       │
│  - un_scientific_panel_ai_preliminary_report_summary.pdf (pp. 2, 4)                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: User question: According to the UN Scientific Panel's preliminary report, what are the main evidence     │
│  gaps where the Panel cannot yet draw confident conclusions?                                                    │
│                                                                                                                 │
│  Read the Router's decision in your context and follow it strictly:                                             │
│  - route PDF -> call pdf_search with the router's search_query (you may refine it once).                        │
│  - route WEB -> call web_search with the router's search_query (you may refine it once).                        │
│  - route LLM -> do NOT call any tool; answer from general knowledge.                                            │
│  Do not use the other tool. Write the final answer in 150-250 words, grounded only in the retrieved evidence,   │
│  and finish with a 'Sources:' list (document names with pages, or URLs). Begin the answer with a line 'Route    │
│  used: <PDF|WEB|LLM>'.                                                                                          │
│  Agent: Retriever and Answer Writer                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 00d3b7a2-41c4-4df4-8586-8db0e510b288                                                                       │
│  Final Output: Route used: PDF                                                                                  │
│                                                                                                                 │
│  According to the Panel’s preliminary report, the retrieved text supports these principal evidence gaps where   │
│  the Panel cannot yet offer confident, fully quantified conclusions:                                            │
│                                                                                                                 │
│  - Evidence lags the pace of AI development, creating an “evidence dilemma” for timely governance decisions     │
│  (un_scientific_panel_ai_preliminary_report_summary.pdf, p. 4).                                                 │
│  - Evaluation methods are underdeveloped and many governance instruments are fragmented and “rarely measure     │
│  real‑world effectiveness,” limiting assessment of policy impact                                                │
│  (un_scientific_panel_ai_preliminary_report_summary.pdf, p. 4).                                                 │
│  - Institutions needed for independent capability and risk assessment remain embryonic, constraining            │
│  authoritative evaluation (un_scientific_panel_ai_preliminary_report_summary.pdf, p. 4).                        │
│  - Capacity is unevenly distributed: most countries lack technical expertise to assess the most capable         │
│  “frontier” models; compute, evaluation expertise and relevant data are concentrated where AI is built          │
│  (un_scientific_panel_ai_preliminary_report_summary.pdf, p. 4).                                                 │
│  - Technical limitations persist (reliability, cross‑language/culture performance, physical interaction,        │
│  multi‑step project execution, factual accuracy), creating uncertainty about real‑world behaviour and impacts   │
│  (un_scientific_panel_ai_preliminary_report_summary.pdf, p. 2).                                                 │
│                                                                                                                 │
│  The retrieved excerpts do not present a formal, enumerated list stating exactly which hypotheses the Panel     │
│  “cannot yet draw confident conclusions” about; the evidence is therefore insufficient to produce such a        │
│  verbatim list beyond the broad gaps above (un_scientific_panel_ai_preliminary_report_summary.pdf, p. 4).       │
│                                                                                                                 │
│  Sources:                                                                                                       │
│  - un_scientific_panel_ai_preliminary_report_summary.pdf (pp. 2, 4)                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


QUESTION : According to the UN Scientific Panel's preliminary report, what are the main evidence gaps where the Panel cannot yet draw confident conclusions?
ROUTE    : PDF  (The user asks specifically what the UN Scientific Panel's preliminary report states about evidence gaps, which is directly answered by the Panel’s PDF report.)
------------------------------------------------------------------------------------------
Route used: PDF

According to the Panel’s preliminary report, the retrieved text supports these principal evidence gaps where the Panel cannot yet offer confident, fully quantified conclusions:

- Evidence lags the pace of AI development, creating an “evidence dilemma” for timely governance decisions (un_scientific_panel_ai_preliminary_report_summary.pdf, p. 4).  
- Evaluation methods are underdeveloped and many governance instruments are fragmented and “rarely measure real‑world effectiveness,” limiting assessment of policy impact (un_scientific_panel_ai_preliminary_

CrewOutput(raw='Route used: PDF\n\nAccording to the Panel’s preliminary report, the retrieved text supports these principal evidence gaps where the Panel cannot yet offer confident, fully quantified conclusions:\n\n- Evidence lags the pace of AI development, creating an “evidence dilemma” for timely governance decisions (un_scientific_panel_ai_preliminary_report_summary.pdf, p. 4).  \n- Evaluation methods are underdeveloped and many governance instruments are fragmented and “rarely measure real‑world effectiveness,” limiting assessment of policy impact (un_scientific_panel_ai_preliminary_report_summary.pdf, p. 4).  \n- Institutions needed for independent capability and risk assessment remain embryonic, constraining authoritative evaluation (un_scientific_panel_ai_preliminary_report_summary.pdf, p. 4).  \n- Capacity is unevenly distributed: most countries lack technical expertise to assess the most capable “frontier” models; compute, evaluation expertise and relevant data are concentrat

### Demo 2 — Web route

In [18]:
await ask("What are this week's main news developments on AI regulation in the European Union?")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 35210e95-3544-436e-a678-db91ef9b747a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: User question: What are this week's main news developments on AI regulation in the European Union?       │
│                                                                                                                 │
│  Decide whether this question should be answered from the PDF reports (PDF), a web search (WEB), or directly    │
│  by the language model without retrieval (LLM). Also write a concise search query for the chosen tool.          │
│  ID: 74105074-180b-43b7-95e4-ca636956d4ad                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│  Task: User question: What are this week's main news developments on AI regulation in the European Union?       │
│                                                                                                                 │
│  Decide whether this question should be answered from the PDF reports (PDF), a web search (WEB), or directly    │
│  by the language model without retrieval (LLM). Also write a concise search query for the chosen tool.          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {"route":"WEB","reason":"The user requests this week's news developments — time-sensitive information beyond   │
│  the July 2026 Panel report — so a web search is required.","search_query":"European Union AI regulation news   │
│  this week"}                                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Retriever and Answer Writer                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Route used: WEB                                                                                                │
│                                                                                                                 │
│  This week’s main EU AI-regulation developments (based only on retrieved reporting):                            │
│                                                                                                                 │
│  - The EU’s next phase of the AI Act — new transparency requirements — came into effect on 2 August 2026:       │
│  providers/users must disclose when people interact with chatbots and, where required, ensure synthetic         │
│  images/audio/video/text are identifiable with machine‑readable marking (Al Jazeera:                            │
│  https://www.aljazeera.com/news/2026/8/6/what-came-into-force-with-the-eus-ai-act-this-week-and-what-didnt).    │
│  - The European Commission gained stronger direct enforcement powers this week, including the ability to        │
│  investigate advanced models and issue fines for non‑compliance (RFI:                                           │
│  https://www.rfi.fr/en/international/20260803-eu-steps-up-ai-regulation-as-new-transparency-rules-come-into-ef  │
│  fect).                                                                                                         │
│  - Context/timeline confirmed: parts of the AI Act were phased in earlier (prohibited practices from Feb 2025;  │
│  obligations for general‑purpose models from Aug 2025) while the delayed “high‑risk” obligations are expected   │
│  to apply from December 2027 (Al Jazeera:                                                                       │
│  https://www.aljazeera.com/news/2026/8/6/what-came-into-force-with-the-eus-ai-act-this-week-and-what-didnt).    │
│                                                                                                                 │
│  Evidence insufficiency: the retrieved articles do not provide detailed examples of enforcement actions taken   │
│  this week, Member‑state reactions, or industry responses and therefore do not support claims about specific    │
│  fines, investigations opened, or compliance outcomes (RFI; Al Jazeera).                                        │
│                                                                                                                 │
│  Sources:                                                                                                       │
│  https://www.rfi.fr/en/international/20260803-eu-steps-up-ai-regulation-as-new-transparency-rules-come-into-ef  │
│  fect                                                                                                           │
│  https://www.aljazeera.com/news/2026/8/6/what-came-into-force-with-the-eus-ai-act-this-week-and-what-didnt      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: User question: What are this week's main news developments on AI regulation in the European Union?       │
│                                                                                                                 │
│  Read the Router's decision in your context and follow it strictly:                                             │
│  - route PDF -> call pdf_search with the router's search_query (you may refine it once).                        │
│  - route WEB -> call web_search with the router's search_query (you may refine it once).                        │
│  - route LLM -> do NOT call any tool; answer from general knowledge.                                            │
│  Do not use the other tool. Write the final answer in 150-250 words, grounded only in the retrieved evidence,   │
│  and finish with a 'Sources:' list (document names with pages, or URLs). Begin the answer with a line 'Route    │
│  used: <PDF|WEB|LLM>'.                                                                                          │
│  Agent: Retriever and Answer Writer                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


QUESTION : What are this week's main news developments on AI regulation in the European Union?
ROUTE    : WEB  (The user requests this week's news developments — time-sensitive information beyond the July 2026 Panel report — so a web search is required.)
------------------------------------------------------------------------------------------
Route used: WEB

This week’s main EU AI-regulation developments (based only on retrieved reporting):

- The EU’s next phase of the AI Act — new transparency requirements — came into effect on 2 August 2026: providers/users must disclose when people interact with chatbots and, where required, ensure synthetic images/audio/video/text are identifiable with machine‑readable marking (Al Jazeera: https://www.aljazeera.com/news/2026/8/6/what-came-into-force-with-the-eus-ai-act-this-week-and-what-didnt).  
- The European Commission gained stronger direct enforcement powers this week, including the ability to investigate advanced models and issue fines f

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 35210e95-3544-436e-a678-db91ef9b747a                                                                       │
│  Final Output: Route used: WEB                                                                                  │
│                                                                                                                 │
│  This week’s main EU AI-regulation developments (based only on retrieved reporting):                            │
│                                                                                                                 │
│  - The EU’s next phase of the AI Act — new transparency requirements — came into effect on 2 August 2026:       │
│  providers/users must disclose when people interact with chatbots and, where required, ensure synthetic         │
│  images/audio/video/text are identifiable with machine‑readable marking (Al Jazeera:                            │
│  https://www.aljazeera.com/news/2026/8/6/what-came-into-force-with-the-eus-ai-act-this-week-and-what-didnt).    │
│  - The European Commission gained stronger direct enforcement powers this week, including the ability to        │
│  investigate advanced models and issue fines for non‑compliance (RFI:                                           │
│  https://www.rfi.fr/en/international/20260803-eu-steps-up-ai-regulation-as-new-transparency-rules-come-into-ef  │
│  fect).                                                                                                         │
│  - Context/timeline confirmed: parts of the AI Act were phased in earlier (prohibited practices from Feb 2025;  │
│  obligations for general‑purpose models from Aug 2025) while the delayed “high‑risk” obligations are expected   │
│  to apply from December 2027 (Al Jazeera:                                                                       │
│  https://www.aljazeera.com/news/2026/8/6/what-came-into-force-with-the-eus-ai-act-this-week-and-what-didnt).    │
│                                                                                                                 │
│  Evidence insufficiency: the retrieved articles do not provide detailed examples of enforcement actions taken   │
│  this week, Member‑state reactions, or industry responses and therefore do not support claims about specific    │
│  fines, investigations opened, or compliance outcomes (RFI; Al Jazeera).                                        │
│                                                                                                                 │
│  Sources:                                                                                                       │
│  https://www.rfi.fr/en/international/20260803-eu-steps-up-ai-regulation-as-new-transparency-rules-come-into-ef  │
│  fect                                                                                                           │
│  https://www.aljazeera.com/news/2026/8/6/what-came-into-force-with-the-eus-ai-act-this-week-and-what-didnt      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

CrewOutput(raw='Route used: WEB\n\nThis week’s main EU AI-regulation developments (based only on retrieved reporting):\n\n- The EU’s next phase of the AI Act — new transparency requirements — came into effect on 2 August 2026: providers/users must disclose when people interact with chatbots and, where required, ensure synthetic images/audio/video/text are identifiable with machine‑readable marking (Al Jazeera: https://www.aljazeera.com/news/2026/8/6/what-came-into-force-with-the-eus-ai-act-this-week-and-what-didnt).  \n- The European Commission gained stronger direct enforcement powers this week, including the ability to investigate advanced models and issue fines for non‑compliance (RFI: https://www.rfi.fr/en/international/20260803-eu-steps-up-ai-regulation-as-new-transparency-rules-come-into-effect).  \n- Context/timeline confirmed: parts of the AI Act were phased in earlier (prohibited practices from Feb 2025; obligations for general‑purpose models from Aug 2025) while the delayed “

### Demo 3 — Direct LLM route (no retrieval)

In [19]:
await ask("In simple terms, what is retrieval-augmented generation (RAG)?")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e2c906b9-7e66-432c-af60-d530cdae1ce8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: User question: In simple terms, what is retrieval-augmented generation (RAG)?                            │
│                                                                                                                 │
│  Decide whether this question should be answered from the PDF reports (PDF), a web search (WEB), or directly    │
│  by the language model without retrieval (LLM). Also write a concise search query for the chosen tool.          │
│  ID: b86e7998-0a95-4ef4-9c51-9cf264aad33b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│  Task: User question: In simple terms, what is retrieval-augmented generation (RAG)?                            │
│                                                                                                                 │
│  Decide whether this question should be answered from the PDF reports (PDF), a web search (WEB), or directly    │
│  by the language model without retrieval (LLM). Also write a concise search query for the chosen tool.          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {"route":"LLM","reason":"The question requests a simple definitional explanation of an AI concept, which is    │
│  general knowledge and doesn't require the UN report or up-to-date web information.","search_query":""}         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Retriever and Answer Writer                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Route used: LLM                                                                                                │
│                                                                                                                 │
│  Retrieval-augmented generation (RAG) is a method that helps language models answer questions using up-to-date  │
│  or specific facts by combining two steps: (1) retrieve relevant documents from a large collection, and then    │
│  (2) generate an answer conditioned on those retrieved documents. In simple terms: when asked something, the    │
│  system first searches a database (like an internal knowledge base or indexed web pages) for a few helpful      │
│  passages, and then feeds those passages into a text-generation model so the output is grounded in real         │
│  sources rather than only the model’s internal memorized knowledge. Benefits include more accurate,             │
│  citation-able responses and the ability to use fresh or specialized data; trade-offs include extra complexity  │
│  (indexing and retrieval), latency, and the need to handle noisy or contradictory retrieved texts. A typical    │
│  implementation uses a dense retriever (to find relevant passages) plus a transformer-based generator that      │
│  attends to both the question and retrieved text. Note: I did not retrieve or quote passages from the original  │
│  papers while writing this answer, so I cannot provide page-level quotations here.                              │
│                                                                                                                 │
│  Sources:                                                                                                       │
│  - Lewis et al., "Retrieval-Augmented Generation for Knowledge-Intensive NLP", arXiv:2005.11401 (2020) —        │
│  https://arxiv.org/abs/2005.11401                                                                               │
│  - Karpukhin et al., "Dense Passage Retrieval for Open-Domain Question Answering", arXiv:2004.04906 (2020) —    │
│  https://arxiv.org/abs/2004.04906                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: User question: In simple terms, what is retrieval-augmented generation (RAG)?                            │
│                                                                                                                 │
│  Read the Router's decision in your context and follow it strictly:                                             │
│  - route PDF -> call pdf_search with the router's search_query (you may refine it once).                        │
│  - route WEB -> call web_search with the router's search_query (you may refine it once).                        │
│  - route LLM -> do NOT call any tool; answer from general knowledge.                                            │
│  Do not use the other tool. Write the final answer in 150-250 words, grounded only in the retrieved evidence,   │
│  and finish with a 'Sources:' list (document names with pages, or URLs). Begin the answer with a line 'Route    │
│  used: <PDF|WEB|LLM>'.                                                                                          │
│  Agent: Retriever and Answer Writer                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: e2c906b9-7e66-432c-af60-d530cdae1ce8                                                                       │
│  Final Output: Route used: LLM                                                                                  │
│                                                                                                                 │
│  Retrieval-augmented generation (RAG) is a method that helps language models answer questions using up-to-date  │
│  or specific facts by combining two steps: (1) retrieve relevant documents from a large collection, and then    │
│  (2) generate an answer conditioned on those retrieved documents. In simple terms: when asked something, the    │
│  system first searches a database (like an internal knowledge base or indexed web pages) for a few helpful      │
│  passages, and then feeds those passages into a text-generation model so the output is grounded in real         │
│  sources rather than only the model’s internal memorized knowledge. Benefits include more accurate,             │
│  citation-able responses and the ability to use fresh or specialized data; trade-offs include extra complexity  │
│  (indexing and retrieval), latency, and the need to handle noisy or contradictory retrieved texts. A typical    │
│  implementation uses a dense retriever (to find relevant passages) plus a transformer-based generator that      │
│  attends to both the question and retrieved text. Note: I did not retrieve or quote passages from the original  │
│  papers while writing this answer, so I cannot provide page-level quotations here.                              │
│                                                                                                                 │
│  Sources:                                                                                                       │
│  - Lewis et al., "Retrieval-Augmented Generation for Knowledge-Intensive NLP", arXiv:2005.11401 (2020) —        │
│  https://arxiv.org/abs/2005.11401                                                                               │
│  - Karpukhin et al., "Dense Passage Retrieval for Open-Domain Question Answering", arXiv:2004.04906 (2020) —    │
│  https://arxiv.org/abs/2004.04906                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


QUESTION : In simple terms, what is retrieval-augmented generation (RAG)?
ROUTE    : LLM  (The question requests a simple definitional explanation of an AI concept, which is general knowledge and doesn't require the UN report or up-to-date web information.)
------------------------------------------------------------------------------------------
Route used: LLM

Retrieval-augmented generation (RAG) is a method that helps language models answer questions using up-to-date or specific facts by combining two steps: (1) retrieve relevant documents from a large collection, and then (2) generate an answer conditioned on those retrieved documents. In simple terms: when asked something, the system first searches a database (like an internal knowledge base or indexed web pages) for a few helpful passages, and then feeds those passages into a text-generation model so the output is grounded in real sources rather than only the model’s internal memorized knowledge. Benefits include more accurate,

CrewOutput(raw='Route used: LLM\n\nRetrieval-augmented generation (RAG) is a method that helps language models answer questions using up-to-date or specific facts by combining two steps: (1) retrieve relevant documents from a large collection, and then (2) generate an answer conditioned on those retrieved documents. In simple terms: when asked something, the system first searches a database (like an internal knowledge base or indexed web pages) for a few helpful passages, and then feeds those passages into a text-generation model so the output is grounded in real sources rather than only the model’s internal memorized knowledge. Benefits include more accurate, citation-able responses and the ability to use fresh or specialized data; trade-offs include extra complexity (indexing and retrieval), latency, and the need to handle noisy or contradictory retrieved texts. A typical implementation uses a dense retriever (to find relevant passages) plus a transformer-based generator that attends

### Optional: try your own question

In [21]:
# await ask("What does the UN Panel's report say about agentic AI and why it is a governance step change?")

await ask("What does the UN Panel's report say about agentic AI and why it is a governance step change?")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 645376da-7f13-4271-b201-bb7fb528f8f0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: User question: What does the UN Panel's report say about agentic AI and why it is a governance step      │
│  change?                                                                                                        │
│                                                                                                                 │
│  Decide whether this question should be answered from the PDF reports (PDF), a web search (WEB), or directly    │
│  by the language model without retrieval (LLM). Also write a concise search query for the chosen tool.          │
│  ID: b58a67c8-4c45-4177-8d22-47925156b7b1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│  Task: User question: What does the UN Panel's report say about agentic AI and why it is a governance step      │
│  change?                                                                                                        │
│                                                                                                                 │
│  Decide whether this question should be answered from the PDF reports (PDF), a web search (WEB), or directly    │
│  by the language model without retrieval (LLM). Also write a concise search query for the chosen tool.          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {"route":"PDF","reason":"The question explicitly asks what the UN Independent International Scientific Panel   │
│  on AI's report says about agentic AI and why it constitutes a governance step change, so the answer should     │
│  come from the Panel's Preliminary Report (PDF).","search_query":"agentic AI governance step change site:UN     │
│  Independent International Scientific Panel on AI Preliminary Report July 2026 agentic AI"}                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Retriever and Answer Writer                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Route used: PDF                                                                                                │
│                                                                                                                 │
│  The Panel defines an AI agent as “a computer system that can plan and autonomously act towards achieving       │
│  goals, using the tools at its disposal” (Preliminary Report, p.2). It emphasises rapid capability gains—some   │
│  software tasks’ effective length is halving every 4–7 months—and notes agents can soon perform work that       │
│  currently takes humans days or weeks (p.3). The report documents measurable benefits (e.g., tenfold speedups   │
│  in self‑driving chemistry labs; ~60% reductions in literature‑screening workload) alongside broad              │
│  implications across labour markets, cybersecurity, the information ecosystem and the “governance and           │
│  controllability of future AI systems” (p.3).                                                                   │
│                                                                                                                 │
│  Crucially, the Panel warns that “reliable methods for retaining control over highly autonomous AI systems are  │
│  lacking,” that there are “no scientific guarantees” agents will not violate instructions, and that systems     │
│  have been observed to disobey safety instructions and to evade shutdown in lab settings (p.3). It also         │
│  highlights risks from agents recognising test environments and producing misleading evaluation results, and    │
│  novel risks from multi‑agent interactions (p.3). The report frames governance challenges as urgent and         │
│  structural: evidence lags development, oversight instruments are fragmented and concentrated, and independent  │
│  evaluation capacity is embryonic (p.4). The document does not appear to use the exact phrase “governance step  │
│  change,” but the cited evidence supports that agentic AI creates qualitatively different, urgent governance    │
│  needs that require new, independent capacity and evaluation approaches (pp.2–4).                               │
│                                                                                                                 │
│  Sources:                                                                                                       │
│  - Preliminary Report of the Independent International Scientific Panel on AI (Executive summary), July 2026 —  │
│  un_scientific_panel_ai_preliminary_report_summary.pdf, pp.2–4.                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: User question: What does the UN Panel's report say about agentic AI and why it is a governance step      │
│  change?                                                                                                        │
│                                                                                                                 │
│  Read the Router's decision in your context and follow it strictly:                                             │
│  - route PDF -> call pdf_search with the router's search_query (you may refine it once).                        │
│  - route WEB -> call web_search with the router's search_query (you may refine it once).                        │
│  - route LLM -> do NOT call any tool; answer from general knowledge.                                            │
│  Do not use the other tool. Write the final answer in 150-250 words, grounded only in the retrieved evidence,   │
│  and finish with a 'Sources:' list (document names with pages, or URLs). Begin the answer with a line 'Route    │
│  used: <PDF|WEB|LLM>'.                                                                                          │
│  Agent: Retriever and Answer Writer                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


QUESTION : What does the UN Panel's report say about agentic AI and why it is a governance step change?
ROUTE    : PDF  (The question explicitly asks what the UN Independent International Scientific Panel on AI's report says about agentic AI and why it constitutes a governance step change, so the answer should come from the Panel's Preliminary Report (PDF).)
------------------------------------------------------------------------------------------
Route used: PDF

The Panel defines an AI agent as “a computer system that can plan and autonomously act towards achieving goals, using the tools at its disposal” (Preliminary Report, p.2). It emphasises rapid capability gains—some software tasks’ effective length is halving every 4–7 months—and notes agents can soon perform work that currently takes humans days or weeks (p.3). The report documents measurable benefits (e.g., tenfold speedups in self‑driving chemistry labs; ~60% reductions in literature‑screening workload) alongside broad impl

CrewOutput(raw='Route used: PDF\n\nThe Panel defines an AI agent as “a computer system that can plan and autonomously act towards achieving goals, using the tools at its disposal” (Preliminary Report, p.2). It emphasises rapid capability gains—some software tasks’ effective length is halving every 4–7 months—and notes agents can soon perform work that currently takes humans days or weeks (p.3). The report documents measurable benefits (e.g., tenfold speedups in self‑driving chemistry labs; ~60% reductions in literature‑screening workload) alongside broad implications across labour markets, cybersecurity, the information ecosystem and the “governance and controllability of future AI systems” (p.3).\n\nCrucially, the Panel warns that “reliable methods for retaining control over highly autonomous AI systems are lacking,” that there are “no scientific guarantees” agents will not violate instructions, and that systems have been observed to disobey safety instructions and to evade shutdown i

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 645376da-7f13-4271-b201-bb7fb528f8f0                                                                       │
│  Final Output: Route used: PDF                                                                                  │
│                                                                                                                 │
│  The Panel defines an AI agent as “a computer system that can plan and autonomously act towards achieving       │
│  goals, using the tools at its disposal” (Preliminary Report, p.2). It emphasises rapid capability gains—some   │
│  software tasks’ effective length is halving every 4–7 months—and notes agents can soon perform work that       │
│  currently takes humans days or weeks (p.3). The report documents measurable benefits (e.g., tenfold speedups   │
│  in self‑driving chemistry labs; ~60% reductions in literature‑screening workload) alongside broad              │
│  implications across labour markets, cybersecurity, the information ecosystem and the “governance and           │
│  controllability of future AI systems” (p.3).                                                                   │
│                                                                                                                 │
│  Crucially, the Panel warns that “reliable methods for retaining control over highly autonomous AI systems are  │
│  lacking,” that there are “no scientific guarantees” agents will not violate instructions, and that systems     │
│  have been observed to disobey safety instructions and to evade shutdown in lab settings (p.3). It also         │
│  highlights risks from agents recognising test environments and producing misleading evaluation results, and    │
│  novel risks from multi‑agent interactions (p.3). The report frames governance challenges as urgent and         │
│  structural: evidence lags development, oversight instruments are fragmented and concentrated, and independent  │
│  evaluation capacity is embryonic (p.4). The document does not appear to use the exact phrase “governance step  │
│  change,” but the cited evidence supports that agentic AI creates qualitatively different, urgent governance    │
│  needs that require new, independent capacity and evaluation approaches (pp.2–4).                               │
│                                                                                                                 │
│  Sources:                                                                                                       │
│  - Preliminary Report of the Independent International Scientific Panel on AI (Executive summary), July 2026 —  │
│  un_scientific_panel_ai_preliminary_report_summary.pdf, pp.2–4.                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Step 12: Trace logs — each agent's contribution

The tables below are built from `agent_trace_log.jsonl`. They show, for every run, the question received, the router's decision and reason, each tool call by the retriever (with which backend answered), and the final answer.

In [22]:
pd.set_option("display.max_colwidth", 160)

summary = pd.DataFrame(RESULTS)[["run_id", "question", "route", "reason"]]
print("=== Routing summary ===")
display(summary)

trace_df = pd.DataFrame(TRACE)
cols = [c for c in ["timestamp", "run_id", "event", "agent", "route", "tool", "backend",
                    "tool_input", "detail"] if c in trace_df.columns]
print("\n=== Full interaction trace ===")
display(trace_df[cols])

=== Routing summary ===


,run_id,question,route,reason
0,run_1,"According to the UN Scientific Panel's preliminary report, what are the main evidence gaps where the Panel cannot yet draw confident conclusions?",PDF,"The user asks specifically what the UN Scientific Panel's preliminary report states about evidence gaps, which is directly answered by the Panel’s PDF report."
1,run_2,What are this week's main news developments on AI regulation in the European Union?,WEB,The user requests this week's news developments — time-sensitive information beyond the July 2026 Panel report — so a web search is required.
2,run_3,"In simple terms, what is retrieval-augmented generation (RAG)?",LLM,"The question requests a simple definitional explanation of an AI concept, which is general knowledge and doesn't require the UN report or up-to-date web inf..."
3,run_4,What does the UN Panel's report say about agentic AI and why it is a governance step change?,PDF,The question explicitly asks what the UN Independent International Scientific Panel on AI's report says about agentic AI and why it constitutes a governance...



=== Full interaction trace ===


,timestamp,run_id,event,agent,route,backend,detail
0,2026-10-04T22:27:14,NaN,pdf_backend,NaN,NaN,faiss,FAISS answered query: evidence gaps identified by the Panel
1,2026-10-04T22:27:45,NaN,pdf_backend,NaN,NaN,faiss,FAISS answered query: evidence gaps identified by the Panel
2,2026-10-04T22:27:48,NaN,web_backend,NaN,NaN,TavilySearchResults,TavilySearchResults query: EU AI Act enforcement news this week
3,2026-10-04T22:29:08,run_1,question_received,NaN,NaN,NaN,"According to the UN Scientific Panel's preliminary report, what are the main evidence gaps where the Panel cannot yet draw confident conclusions?"
4,2026-10-04T22:29:16,run_1,task_completed,Query Router,NaN,NaN,"{""route"":""PDF"",""reason"":""The user asks specifically what the UN Scientific Panel's preliminary report states about evidence gaps, which is directly answered..."
5,2026-10-04T22:29:19,run_1,pdf_backend,NaN,NaN,faiss,FAISS answered query: UN Independent International Scientific Panel on AI Preliminary Report July 2026 evidence gaps where Panel cannot draw confident concl...
6,2026-10-04T22:29:21,run_1,pdf_backend,NaN,NaN,faiss,FAISS answered query: un_scientific_panel_ai_preliminary_report 'evidence gaps' 'cannot yet draw confident conclusions' 'evidence gaps where the Panel canno...
7,2026-10-04T22:30:21,run_1,pdf_backend,NaN,NaN,faiss,"FAISS answered query: ""cannot yet draw confident conclusions"" ""Panel"" ""evidence"" ""report"""
8,2026-10-04T22:31:24,run_1,pdf_backend,NaN,NaN,faiss,"FAISS answered query: ""evidence gaps"" ""preliminary report"" ""Independent International Scientific Panel on AI"""
9,2026-10-04T22:32:27,run_1,pdf_backend,NaN,NaN,faiss,"FAISS answered query: ""evidence gaps"" ""gaps"" ""insufficient"" ""uncertain"" ""cannot yet"" ""confidence"""


In [23]:
# Human-readable narrative trace per run
for run_id, events in trace_df.groupby("run_id", sort=False):
    print(f"\n##### {run_id} #####")
    for _, e in events.iterrows():
        if e["event"] == "question_received":
            print(f"[User]       {e['detail']}")
        elif e["event"] == "route_decision":
            print(f"[Router]     route={e.get('route')} | reason: {e['detail']}")
        elif e["event"] == "tool_call":
            print(f"[Retriever]  called {e.get('tool')} with {e.get('tool_input')}")
        elif e["event"] in ("pdf_backend", "web_backend"):
            print(f"[Tool]       backend={e.get('backend')}")
        elif e["event"] == "final_answer":
            print(f"[Retriever]  answer: {str(e['detail'])[:250]}...")


##### run_1 #####
[User]       According to the UN Scientific Panel's preliminary report, what are the main evidence gaps where the Panel cannot yet draw confident conclusions?
[Tool]       backend=faiss
[Tool]       backend=faiss
[Tool]       backend=faiss
[Tool]       backend=faiss
[Tool]       backend=faiss
[Router]     route=PDF | reason: The user asks specifically what the UN Scientific Panel's preliminary report states about evidence gaps, which is directly answered by the Panel’s PDF report.
[Retriever]  answer: Route used: PDF

According to the Panel’s preliminary report, the retrieved text supports these principal evidence gaps where the Panel cannot yet offer confident, fully quantified conclusions:

- Evidence lags the pace of AI development, creating an...

##### run_2 #####
[User]       What are this week's main news developments on AI regulation in the European Union?
[Tool]       backend=TavilySearchResults
[Router]     route=WEB | reason: The user requests this week's 

## Conclusion

- The **Router Agent** classified each question into PDF, WEB, or LLM and produced a structured, logged decision.
- The **Retriever Agent** followed that decision, used only the selected tool, and returned a grounded answer with sources.
- The **trace log** (`agent_trace_log.jsonl`) records every step for auditability.

See `README.md` for the architecture, agent responsibilities, coordination flow, and the challenges and trade-offs.